# EoloSense · reprodução no Google Colab

Detecção precoce de falhas em turbinas eólicas com o dataset **CARE to Compare** (Fraunhofer IEE, Zenodo).
Este notebook baixa os dados, roda os testes e treina **Árvore de Decisão, Ensembles, SVM e Regressão Linear**
com o mesmo protocolo do K-means. Nada precisa ser instalado na sua máquina.

**Tempo:** o download tem 5,5 GB. Com só a fazenda A (22 eventos), o treino dos quatro modelos leva da ordem de
uma hora nos 2 núcleos do Colab. As três fazendas (95 eventos) levam várias horas: prefira rodar localmente com `--jobs`.

Detalhes do método: `README.md` e `Comum/README.md` do repositório.

## 1. Código e dependências

In [ ]:
!git clone -q https://github.com/MiguelFirme/EoloSense-MachineLearning.git
%cd EoloSense-MachineLearning
!pip -q install -r requirements.txt

## 2. Escolha o que rodar

In [ ]:
FAZENDAS = "A"        # "A" (rápido, 22 eventos), ou "ABC" (todos os 95 eventos)
MODELOS = "arvore ensembles svm linear"
EXECUCAO = "colab"    # nome da pasta de resultados (outputs/<EXECUCAO>)

## 3. Dataset CARE to Compare (v6)

Baixa uma vez, confere o MD5 publicado e extrai só as fazendas escolhidas.

In [ ]:
import hashlib, os, pathlib
zip_path = pathlib.Path("data/CARE_To_Compare.zip")
zip_path.parent.mkdir(exist_ok=True)
if not zip_path.exists():
    !wget -q --show-progress -c -O data/CARE_To_Compare.zip "https://zenodo.org/records/15846963/files/CARE_To_Compare.zip?download=1"

md5 = hashlib.md5()
with open(zip_path, "rb") as f:
    for block in iter(lambda: f.read(1 << 24), b""):
        md5.update(block)
assert md5.hexdigest() == "2547b58c21ac8c242d13232860cf500c", "Download corrompido: apague o zip e rode de novo."
print("MD5 conferido")

In [ ]:
for farm in FAZENDAS:
    !unzip -q -n data/CARE_To_Compare.zip "*Wind Farm {farm}/*" -d data/CARE_To_Compare
!find data/CARE_To_Compare -name "event_info.csv"

## 4. Testes automatizados

Conferem as garantias do método (sem vazamento, atributos causais, escore aponta o sensor certo) num mini-dataset sintético.

In [ ]:
!python -m unittest discover -s Comum/tests -v

## 5. Treino e avaliação

Cada evento ganha seu próprio modelo, treinado só com operação normal. A regra de alerta é escolhida nos eventos de desenvolvimento e congelada antes dos eventos reservados. Se a sessão cair, rode a célula de novo: os eventos prontos são pulados.

In [ ]:
!python Comum/src/treinar_modelos.py --farms {FAZENDAS} --models {MODELOS} --run {EXECUCAO} --jobs 2

## 6. Resultados

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open(f"Comum/outputs/{EXECUCAO}/COMPARACAO.md", encoding="utf-8").read()))
display(Image(f"Comum/outputs/{EXECUCAO}/comparacao.png"))

In [ ]:
pastas = {"arvore": "Arvore-de-Decisao", "ensembles": "Ensembles", "svm": "SVM", "linear": "Regressao-Linear"}
for chave in MODELOS.split():
    raiz = f"{pastas[chave]}/outputs/{EXECUCAO}"
    display(Markdown(open(f"{raiz}/RESULTADOS.md", encoding="utf-8").read()))
    display(Image(f"{raiz}/figuras/linha_do_tempo_reservados.png"))

## 7. Baixar os resultados

Compacta as pastas `outputs/` (modelos, previsões e relatórios) para usar no painel Streamlit local ou no relatório.

In [ ]:
!zip -q -r resultados_{EXECUCAO}.zip */outputs/{EXECUCAO} Comum/outputs/{EXECUCAO}
from google.colab import files
files.download(f"resultados_{EXECUCAO}.zip")